# OsteoClassAI v2 · 07 Mejora 2: robustez ante otros equipos y recortes

Corrida **C**: misma receta que B (ConvNeXt-Tiny, SORD + TTA, 288 px), con `--aug robust`:
- alejar la rodilla hasta 0,6× (con bordes negros) y acercarla hasta 1,15×;
- franjas negras arriba y abajo, como en los recortes anchos rellenados a cuadrado;
- gamma, brillo y contraste más amplios, nitidez y desenfoque.

El modelo se elige con la validación de Chen y la prueba de Chen se evalúa una vez. Después se mide en Gornale, **que ya no es una prueba imparcial** (estos aumentos se diseñaron viendo sus resultados): sirve para ver la tendencia, no para afirmar generalización.

**Antes de ejecutar:** `Digital Knee X-ray Images.zip` debe seguir en `MyDrive/OsteoClassAI/datos_externos/`.

*Entorno de ejecución → Cambiar tipo → GPU (T4)*, y luego *Ejecutar todas* (~25 min).

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if os.path.isdir('/content/osteoclassai'):
    !cd /content/osteoclassai && git pull -q
else:
    !git clone -q https://github.com/walterandres23/osteoclassai.git /content/osteoclassai
%cd /content/osteoclassai/v2
!pip -q install -e . timm kagglehub imagehash

In [ ]:
import kagglehub, pathlib
path = pathlib.Path(kagglehub.dataset_download('shashwatwork/knee-osteoarthritis-dataset-with-severity'))
root = next(p.parent for p in path.rglob('train') if p.is_dir())
!python -m osteoclass.data --root "$root" --out outputs/data > data.log && tail -n 3 data.log

## Vista previa de los aumentos (12 versiones de una misma rodilla de entrenamiento)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, torch
from PIL import Image
from osteoclass.dataset import build_transforms, IMAGENET_MEAN, IMAGENET_STD
df = pd.read_csv('outputs/data/splits.csv'); r = df[(df.split == 'train') & (df.kl == 2)].iloc[0]
img = Image.open(pathlib.Path(root) / r.path.replace('\\', '/')).convert('L')
tf = build_transforms(288, train=True, aug='robust')
m, s = torch.tensor(IMAGENET_MEAN)[:, None, None], torch.tensor(IMAGENET_STD)[:, None, None]
fig, ax = plt.subplots(2, 6, figsize=(15, 5.4))
for a in ax.flat:
    a.imshow((tf(img) * s + m)[0].clamp(0, 1), cmap='gray'); a.axis('off')
plt.tight_layout(); plt.show()

## C · SORD + TTA, 288 px, aumentos de robustez

In [ ]:
!python -m osteoclass.train --root "$root" --splits outputs/data/splits.csv --model convnext_tiny --loss sord --tta --size 288 --aug robust --epochs 30 --patience 8 --out outputs/runs/convnext_tiny_sord_288_robust
!mkdir -p /content/drive/MyDrive/OsteoClassAI/runs && cp -r outputs/runs/convnext_tiny_sord_288_robust /content/drive/MyDrive/OsteoClassAI/runs/

## Gornale (ya no imparcial): B frente a C

In [ ]:
ZIP = '/content/drive/MyDrive/OsteoClassAI/datos_externos/Digital Knee X-ray Images.zip'
!mkdir -p /content/gornale && cd /content/gornale && unzip -q -o "$ZIP"
!which unrar > /dev/null || apt-get -qq install -y unrar > /dev/null
!cd "/content/gornale/Digital Knee X-ray Images/Knee X-ray Images" && for f in *.rar; do unrar x -o+ -idq "$f"; done
groot = '/content/gornale/Digital Knee X-ray Images/Knee X-ray Images'
RUNS = '/content/drive/MyDrive/OsteoClassAI/runs'
!python -m osteoclass.external --root "$groot" --runs $RUNS/convnext_tiny_sord_288 $RUNS/convnext_tiny_sord_288_robust --exploratory --out outputs/external/gornale_robustez
!mkdir -p /content/drive/MyDrive/OsteoClassAI/external && cp -r outputs/external/gornale_robustez /content/drive/MyDrive/OsteoClassAI/external/